In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/K2Debug/Financial-News-Sentiment-Analyzer-for-Economic-Forecasting.git"
PROJECT_ROOT = Path("/content/EF-02")
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not PROJECT_ROOT.exists():
        print("Workspace not found — cloning repo and installing dependencies...")
        !git clone {REPO_URL} {PROJECT_ROOT}
        !pip install -q python-dotenv groq openai bs4
        !pip install -q -r /content/EF-02/requirements.txt
        print("Workspace ready.")
    os.chdir(NOTEBOOKS_DIR)

# 01. Headlines Scraper

Collects financial news headlines from Daily News, The Citizen, and IPP Media.

In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import os
import time
from datetime import datetime

## Shared Functions

In [7]:
def parse_date(raw_date):
    if not raw_date:
        return "unknown"
    cleaned = raw_date.split("-")[0].strip()
    for fmt in ("%B %d, %Y", "%b %d, %Y"):
        try:
            return datetime.strptime(cleaned, fmt).strftime("%Y-%m-%d")
        except ValueError:
            pass
    return cleaned


def get_article_urls(page_html, config):
    soup = BeautifulSoup(page_html, "html.parser")
    results = []

    container = None
    if config["article_container"]:
        container = soup.select_one(config["article_container"])

    if container:
        search_scope = container
        selector = config["article_link_sel"]
    else:
        selector = (
            config["fallback_link_sel"]
            if config["article_container"]
            else config["article_link_sel"]
        )
        search_scope = soup

    cards = search_scope.select(selector)
    if not cards:
        return results

    for card in cards:
        if config["category_filter"]:
            category_tag = card.select_one("span.article-topic")
            if not category_tag:
                continue
            if category_tag.get_text(strip=True).lower() not in config["category_filter"]:
                continue

        if card.name == "a":
            relative_url = card.get("href", "").strip()
        else:
            inner_link = card.find("a")
            relative_url = inner_link.get("href", "").strip() if inner_link else ""

        if not relative_url:
            continue

        full_url = config["base_domain"] + relative_url

        headline = card.get_text(strip=True)
        if config["name"] == "The Citizen":
            h3 = card.select_one("h3.title-small")
            headline = h3.get_text(strip=True) if h3 else headline
        if config["name"] == "IPP Media":
            h3 = card.select_one("h3")
            headline = h3.get_text(strip=True) if h3 else headline

        if not headline:
            continue

        date = "unknown"
        parent = card.find_parent(["article", "li", "div"])
        if parent:
            date_tag = parent.select_one(config["date_sel"])
            if date_tag:
                date = parse_date(date_tag.get_text(strip=True))

        results.append({"headline": headline, "url": full_url, "date": date})

    return results


def append_to_csv(records, filename):
    if not records:
        return
    file_exists = os.path.isfile(filename)
    with open(filename, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["date", "headline", "url", "source", "scraped_on"])
        if not file_exists:
            writer.writeheader()
        writer.writerows(records)


def scrape_site(config, start_page, end_page, output_file, checkpoint_every=30):
    print(f"Starting scrape : {config['name']}")
    print(f"Pages           : {start_page} -> {end_page}")
    print(f"Output          : {output_file}")
    print()

    scraped_on    = datetime.today().strftime("%Y-%m-%d")
    total_saved   = 0
    batch_records = []
    scraped_pages = []
    all_skipped   = []
    all_failed    = []

    for page_num in range(start_page, end_page + 1):
        page_url = config["base_url"].format(page_num)

        try:
            response = requests.get(
                page_url,
                timeout=15,
                headers={"User-Agent": "Mozilla/5.0 (compatible; research-scraper/1.0)"}
            )
            if response.status_code == 404:
                print(f"404 at page {page_num} -- end of archive, stopping")
                break
            response.raise_for_status()

        except requests.RequestException as e:
            all_failed.append(f"page {page_num} ({type(e).__name__})")
            continue

        articles = get_article_urls(response.text, config)

        if not articles:
            all_skipped.append(f"page {page_num} (no articles)")
            continue

        for article in articles:
            batch_records.append({
                "date"      : article["date"],
                "headline"  : article["headline"],
                "url"       : article["url"],
                "source"    : config["name"],
                "scraped_on": scraped_on,
            })

        total_saved += len(articles)
        scraped_pages.append(f"page {page_num}")

        # Live overwrite -- shows running list of scraped pages on one line
        print(f"{total_saved} headlines scraped: {', '.join(scraped_pages)}", end="\r", flush=True)

        if page_num % checkpoint_every == 0:
            append_to_csv(batch_records, output_file)
            batch_records = []
            # Permanently print the scraped line then reset for next batch
            print(f"{total_saved} headlines scraped: {', '.join(scraped_pages)}")
            print(f"Checkpoint saved -- page {page_num} | {output_file}")
            print()
            scraped_pages = []

        time.sleep(1)

    if batch_records:
        append_to_csv(batch_records, output_file)

    # Flush any remaining scraped pages not yet permanently printed
    if scraped_pages:
        print(f"[{total_saved} headlines scraped:] {', '.join(scraped_pages)}")

    # Skipped and failed summary at the end
    if all_skipped:
        print(f"Skipped : {', '.join(all_skipped)}")
    if all_failed:
        print(f"Failed  : {', '.join(all_failed)}")

    print()
    print(f"Scrape complete!")
    print(f"Total headlines saved : {total_saved}")
    print(f"File                  : {output_file}")


def test_scrape(config, start_page, num_pages=3):
    print(f"Test scrape : {config['name']}")
    print(f"Pages       : {start_page} -> {start_page + num_pages - 1}")
    print()

    for page_num in range(start_page, start_page + num_pages):
        page_url = config["base_url"].format(page_num)
        print(f"Page {page_num}: {page_url}")

        try:
            response = requests.get(
                page_url,
                timeout=15,
                headers={"User-Agent": "Mozilla/5.0 (compatible; research-scraper/1.0)"}
            )
            response.raise_for_status()
        except requests.RequestException as e:
            print(f"  Failed: {e}")
            continue

        articles = get_article_urls(response.text, config)

        if not articles:
            print("  No articles found -- check selectors")
        else:
            print(f"  {len(articles)} articles found:")
            for a in articles[:5]:
                print(f"    [{a['date']}] {a['headline'][:80]}")

        print()
        time.sleep(1)


## Daily News

In [3]:
DAILY_NEWS = {
    "name"             : "Daily News",
    "base_url"         : "https://dailynews.co.tz/category/business/page/{}/",
    "article_container": "#masonry-grid",
    "article_link_sel" : "h2.entry-title a",
    "fallback_link_sel": "h2 a, h3 a",
    "date_sel"         : "span.date",
    "category_filter"  : [],
    "base_domain"      : "",
}

# Test: confirm selectors work before running the full scrape
# Adjust start_page to any page in your target range
test_scrape(DAILY_NEWS, start_page=287, num_pages=3)

Test scrape : Daily News
Pages       : 287 -> 289

Page 287: https://dailynews.co.tz/category/business/page/287/
  10 articles found:
    [2025-02-28] Kariakoo officially starts 24-hour business operation
    [2025-02-28] Tanzania’s microfinance sector sees explosive growth
    [2025-02-28] Alabuga Start Programme to empower Gen-Z girls in Tanzania
    [2025-02-28] Gigantic 62tri/- investment boom in four years
    [2025-02-28] TZ boosts gold reserves with domestic purchase programme

Page 288: https://dailynews.co.tz/category/business/page/288/
  10 articles found:
    [2025-02-27] BoT to deepen collaboration with media
    [2025-02-27] OTR, CAG to collaborate in overseeing public corporations
    [2025-02-27] New dawn for industrial sugar
    [2025-02-26] Samia hands over 35 boats to boost fishing in Tanga
    [2025-02-26] Morogoro RC calls for strategic investment areas to boost tourism, economic grow

Page 289: https://dailynews.co.tz/category/business/page/289/
  10 articles found

In [11]:
# Daily News full scrape
# Pages 287-683 cover 2022-2024 (from previous run)
# To resume after a crash: update start_page to last successful checkpoint page

scrape_site(
    config          = DAILY_NEWS,
    start_page      = 314,
    end_page        = 710,
    output_file     = "../data/raw/dailynews_raw.csv",
    checkpoint_every= 30,
)

Starting scrape : Daily News
Pages           : 314 -> 710
Output          : ../data/raw/dailynews_raw.csv

170 headlines scraped: page 314, page 315, page 316, page 317, page 318, page 319, page 320, page 321, page 322, page 323, page 324, page 325, page 326, page 327, page 328, page 329, page 330
Checkpoint saved -- page 330 | ../data/raw/dailynews_raw.csv

750 headlines scraped: page 331, page 332, page 333, page 334, page 335, page 336, page 337, page 338, page 339, page 340, page 341, page 342, page 343, page 344, page 345, page 346, page 347, page 348, page 349, page 350, page 351, page 352, page 353, page 354, page 355, page 356, page 357, page 358, page 361, page 362, page 363, page 364, page 365, page 366, page 367, page 368, page 369, page 370, page 371, page 372, page 373, page 374, page 375, page 376, page 377, page 378, page 379, page 380, page 381, page 382, page 383, page 384, page 385, page 386, page 387, page 388, page 389, page 390
Checkpoint saved -- page 390 | ../dat

## The Citizen

In [12]:
THE_CITIZEN = {
    "name"             : "The Citizen",
    "base_url"         : "https://www.thecitizen.co.tz/service/search/tanzania/2718734?pageNum={}&query=business&sortByDate=true",
    "article_container": None,
    "article_link_sel" : "a.teaser-image-left",
    "fallback_link_sel": "h3 a",
    "date_sel"         : "span.date",
    "category_filter"  : ["business", "national"],
    "base_domain"      : "https://www.thecitizen.co.tz",
}

# Test: confirm selectors work before running the full scrape
# Adjust start_page to any page in your target range
test_scrape(THE_CITIZEN, start_page=424, num_pages=3)

Test scrape : The Citizen
Pages       : 424 -> 426

Page 424: https://www.thecitizen.co.tz/service/search/tanzania/2718734?pageNum=424&query=business&sortByDate=true
  7 articles found:
    [2024-02-20] Tanzania beats Kenya and Uganda in economic growth
    [2024-02-20] Human rights centre to help Tanzanians sue the government over power rationing
    [2024-02-20] CRDB Bank organizes seminar on financial inclusion for journalists
    [2024-02-20] Tanzania now turns to Egyptian to invest in berths 13-15 at Dar port
    [2024-02-19] 40 graduates join GGML’s new internship project

Page 425: https://www.thecitizen.co.tz/service/search/tanzania/2718734?pageNum=425&query=business&sortByDate=true
  4 articles found:
    [2024-02-19] Dar bourse trading falls on absence of foreign investors
    [2024-02-18] Non-tariff barriers cost region $16 million, threatening intra-EAC trade
    [2024-02-18] Africa lending groups lose appeal to global banks
    [2024-02-17] Tanzania in crisis: No electrici

In [13]:
scrape_site(
    config          = THE_CITIZEN,
    start_page      = 290,
    end_page        = 867,
    output_file     = "../data/raw/citizen_raw.csv",
    checkpoint_every= 30,
)

Starting scrape : The Citizen
Pages           : 290 -> 867
Output          : ../data/raw/citizen_raw.csv

52 headlines scraped: page 290, page 291, page 292, page 293, page 294, page 295, page 296, page 297, page 298, page 299, page 300
Checkpoint saved -- page 300 | ../data/raw/citizen_raw.csv

211 headlines scraped: page 301, page 302, page 303, page 304, page 305, page 306, page 307, page 308, page 309, page 310, page 311, page 312, page 313, page 314, page 315, page 316, page 317, page 318, page 319, page 320, page 321, page 322, page 323, page 324, page 325, page 326, page 327, page 328, page 329, page 330
Checkpoint saved -- page 330 | ../data/raw/citizen_raw.csv

402 headlines scraped: page 331, page 332, page 333, page 334, page 335, page 336, page 337, page 338, page 339, page 340, page 341, page 342, page 343, page 344, page 345, page 346, page 347, page 348, page 349, page 350, page 351, page 352, page 353, page 354, page 355, page 356, page 357, page 358, page 359, page 360